# M8.3E — Architecture freeze

Plan: [`plans/milestone_08/08_single_view_localization_plan.md`](../../plans/milestone_08/08_single_view_localization_plan.md).  
**Not in Final Report** (assumed via `m8_single_view_block_freeze()` in M9).

Writes `win3e_architecture_freeze.json` and optionally runs a confirmatory triad train/val pass.


In [1]:
from pathlib import Path
import shutil
import sys
import subprocess

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Could not locate repository root containing pyproject.toml")
    ROOT = ROOT.parent

# Concurrent notebook installs race on ./build (Errno 17 File exists).
for name in ("build", "dist"):
    shutil.rmtree(ROOT / name, ignore_errors=True)
for egg in (ROOT / "src").glob("*.egg-info"):
    shutil.rmtree(egg, ignore_errors=True)

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-cache-dir",
        f"{ROOT}[dl,dev]",
        "-c",
        str(ROOT / "requirements.txt"),
    ]
)

from gummybear.paths import display_path

print(f"ROOT={display_path(ROOT)}")



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
ROOT=.


In [2]:
from tomography_ml_validation.milestone_08 import m8_corpus_paths, describe_paths

DATA_MODE = "full" if (ROOT / "data/generated/m8_1/single_particle").is_dir() else "demo"
paths = m8_corpus_paths(ROOT, data_mode=DATA_MODE)
print(f"DATA_MODE={DATA_MODE}")
print(describe_paths(paths))


DATA_MODE=full
workbook_path=configs/m8/localization_single_particle.xlsx  output_root=data/generated/m8_1/single_particle  cache_root=data/generated/m8_1/single_particle/_cache


In [3]:
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import torch

from tomography_ml import get_device
from tomography_ml.gummybear_data_catalog.task_dataset import build_task_dataset
from tomography_ml.studies import M8_CANONICAL_LR_BY_ARCH
import tomography_ml_validation.milestone_08.validation as m8_validation
from gummybear_validation.notebook_tools import run_installed_pytest_test

from tomography_ml_validation.milestone_08 import DEFAULT_N_REPEAT

SMOKE = True
RUN_STUDY = None  # None = auto (run when runs CSV missing)
N_REPEAT = DEFAULT_N_REPEAT
BASE_SEED = 0
NUM_EPOCHS = 40 if SMOKE else 200
EARLY_STOP = 25 if SMOKE else 40

device = get_device()
print(f"device={device}")
print("canonical LRs", M8_CANONICAL_LR_BY_ARCH)


device=mps
canonical LRs {'pooled': 0.001, 'fourier': 0.03, 'flatten': 0.0003}


In [4]:
from tomography_ml.localization import win3e_architecture_freeze
from tomography_ml_validation.milestone_08 import (
    DEFAULT_N_REPEAT,
    freeze_record_dataframe,
    load_win3e_supporting_results,
    run_win3e_confirmatory_triad,
    win3e_results_dir,
    win3e_task_specs,
    win3_grids_summary,
    write_win3e_freeze_artifacts,
)

results_dir = win3e_results_dir(ROOT)


## Milestone 8 Step 3E freeze record

In [5]:
display(freeze_record_dataframe(win3e_architecture_freeze()))
display(win3_grids_summary()['3E_controls'])


,selected_variant,spatial_readout_type,depth,widths,downsampling,head,head_hidden,positive_baseline,negative_control,library_class,selection_rationale,accuracy_complexity_notes,freeze_fields
0,fourier_base_mlp,fourier_coded_pool,3,"(16, 32, 64)",base,mlp,128,flatten_base_mlp,pooled_base_base,LocalizerSingleViewFourier,parameter_economy_vs_performance_after_win3a_3...,pool_negative_control;fourier_primary_compact_...,"(cnn_depth, channel_widths, downsampling_geome..."


,arch_name,head_type,encoder_channels,downsample,maxpool_after_blocks,pool_schedule,pre_flatten_channels,embed_dim,flatten_hidden,flatten_head,input_representation,normalisation
0,fourier_base_mlp,fourier,"(16, 32, 64)",base,none,"3× (Conv3×3 → ReLU), no MaxPool",None,128,128,mlp,anomaly_ref,none
1,flatten_base_mlp,flatten,"(16, 32, 64)",base,none,"3× (Conv3×3 → ReLU), no MaxPool",None,128,128,mlp,anomaly_ref,none
2,pooled_base_base,pooled,"(16, 32, 64)",base,none,"3× (Conv3×3 → ReLU), no MaxPool",None,128,128,mlp,anomaly_ref,none


## Contract test

In [6]:
run_installed_pytest_test(m8_validation, "test_m8_3e_architecture_freeze_records_fourier_base_mlp")


M8.3E
Test executed: test_m8_3e_architecture_freeze_records_fourier_base_mlp()

pytest:
../../venv/lib/python3.12/site-packages/tomography_ml_validation/milestone_08/validation.py . [100%]
============================== 1 passed in 1.99s ===============================

Test proves: Milestone 8 Step 3E architecture freeze records Fourier-base + MLP as primary.


## Supporting 3C / 3D CSVs + freeze artefact

In [7]:
support = load_win3e_supporting_results(ROOT)
for win, df in support.items():
    print(f"M8 Step {win}:", "missing" if df is None else f"{len(df)} rows")
paths = write_win3e_freeze_artifacts(ROOT, channel_capacity_df=support['3C'], head_expressiveness_df=support['3D'])
print("Wrote", display_path(paths['freeze_json']))


M8 Step 3C: 5 rows
M8 Step 3D: 5 rows
Wrote checkpoints/m8/m08_3e_architecture_freeze/win3e_architecture_freeze.json


## Optional confirmatory triad (train / val)

In [8]:
from tomography_ml_validation.milestone_08 import load_m8_catalog_rows

RUN_CONFIRM = False
if RUN_CONFIRM:
    rows = load_m8_catalog_rows(ROOT, data_mode=DATA_MODE)
    train_task, val_task, _ = win3e_task_specs()
    train_ds = build_task_dataset(rows, train_task)
    val_ds = build_task_dataset(rows, val_task)
    summary_df, runs_df, _ = run_win3e_confirmatory_triad(
        train_ds, val_ds, train_task,
        device=device,
        num_epochs=NUM_EPOCHS,
        early_stop_patience=EARLY_STOP,
        results_dir=results_dir,
        n_repeat=N_REPEAT,
    )
    display(summary_df)
else:
    print("Set RUN_CONFIRM=True to train the frozen interpretive triad.")


Set RUN_CONFIRM=True to train the frozen interpretive triad.
